# NVIDIA CUTLASS Code Analysis & Hidden Files Discovery

This notebook provides a deep analysis of code from the cloned **NVIDIA CUTLASS** repository alongside the repository's custom automation tooling, and executes a search for all hidden files within the CUTLASS project.

--- 
## 1. NVIDIA CUTLASS Code Analysis

### 1.1 Embedded CUTLASS C++ GEMM Kernel (`cutlass/examples/00_basic_gemm/basic_gemm.cu`)

CUTLASS (CUDA Template Library for Dense Linear Algebra) relies on modern C++ template metaprogramming to generate high-performance CUDA kernels. Below is the basic GEMM implementation from the cloned CUTLASS repository:

In [3]:
from pathlib import Path

# Read and embed CUTLASS C++ GEMM code
cutlass_cpp_path = Path('cutlass/examples/00_basic_gemm/basic_gemm.cu')
cutlass_cpp_code = cutlass_cpp_path.read_text()

print(f"=== CUTLASS C++ Code Loaded: {len(cutlass_cpp_code)} characters ===")
# Display snippet of CUTLASS kernel definition
snippet_lines = [line for line in cutlass_cpp_code.splitlines() if line.strip() and not line.strip().startswith('*') and not line.strip().startswith('/*') and not line.strip().startswith('//')][:25]
print("\n".join(snippet_lines))

=== CUTLASS C++ Code Loaded: 14698 characters ===
  This example demonstrates how to call a CUTLASS GEMM kernel and provides a naive reference
  matrix multiply kernel to verify its correctness.
  The CUTLASS Gemm template is instantiated in the function CutlassSgemmNN. This is kernel computes
  the general matrix product (GEMM) using single-precision floating-point arithmetic and assumes
  all matrices have column-major layout.
  The threadblock tile size is chosen as 128x128x8 which offers good performance for large matrices.
  See the CUTLASS Parallel for All blog post for more exposition on the tunable parameters available
  in CUTLASS.
  https://devblogs.nvidia.com/cutlass-linear-algebra-cuda/
  Aside from defining and launching the SGEMM kernel, this example does not use any other components
  or utilities within CUTLASS. Such utilities are demonstrated elsewhere in other examples and are
  prevalent in the CUTLASS unit tests.
  This example has delibrately been kept similar to t

#### Technical Analysis of CUTLASS C++ GEMM:
1. **Template Specialization (`cutlass::gemm::device::Gemm`)**:
   - Specifies data types (e.g. `float` for single precision SGEMM).
   - Configures memory layouts (`cutlass::layout::ColumnMajor` or `RowMajor`).
   - Sets threadblock tile size (`128x128x8`), warp shape, and instruction tile shape to maximize compute unit throughput.
2. **Epilogue Functor (`cutlass::epilogue::thread::LinearCombination`)**:
   - Computes $D = \alpha \cdot (A \times B) + \beta \cdot C$.
3. **Zero-Overhead Abstraction**:
   - Uses host-side `Gemm::Arguments` struct to package device pointers and dimensions ($M, N, K$) before launching the CUDA kernel via `gemm_op()`.

### 1.2 Embedded CUTLASS Python Interface (`cutlass/examples/40_cutlass_py/gemm.py`)

CUTLASS provides Python bindings allowing users to construct, compile, and run GEMM operations directly from Python using NumPy or PyTorch tensors:

In [6]:
# Read and embed CUTLASS Python GEMM example
cutlass_py_path = Path('cutlass/examples/40_cutlass_py/gemm.py')
cutlass_py_code = cutlass_py_path.read_text()

print(f"=== CUTLASS Python Interface Code Loaded: {len(cutlass_py_code)} characters ===")
py_snippet = [line for line in cutlass_py_code.splitlines() if 'Gemm' in line or 'pycutlass' in line or 'TileDescription' in line][:15]
print("\n".join(py_snippet))

=== CUTLASS Python Interface Code Loaded: 6047 characters ===
import cutlass.backend as pycutlass
pycutlass.get_memory_pool(init_pool_size=2**30, max_pool_size=2**32)
pycutlass.compiler.nvcc()
tile_description = TileDescription(
epilogue_functor = pycutlass.LinearCombination(C.element, C.alignment, element_acc, element_epilogue)
operation = GemmOperationUniversal(
pycutlass.compiler.add_module(operations)
problem_size = cutlass_bindings.gemm.GemmCoord(args.m, args.n, args.k)
arguments = GemmArguments(


#### Technical Analysis of CUTLASS Python Bindings:
- **`TensorDescription`**: Encapsulates data type (`float16`, `float32`), layout (`ColumnMajor`/`RowMajor`), and memory alignment requirements.
- **`TileDescription`**: Defines threadblock shape (e.g., `[128, 128, 32]`), pipeline stages, and warp layout.
- **`GemmOperationUniversal`**: Generates and compiles C++/CUDA source via `nvcc` at runtime and dispatches execution dynamically.

--- 
## 2. User Tooling Analysis: `scripts/lm_studio_task.py`

Below is the user's custom script embedded and analyzed:

In [9]:
# Read and embed user script
user_script_path = Path('scripts/lm_studio_task.py')
user_code = user_script_path.read_text()

print(f"=== User Script Loaded ({len(user_code)} chars) ===")
print("\n".join(user_code.splitlines()[:15]))

=== User Script Loaded (2229 chars) ===
#!/usr/bin/env python3
"""Run a GitHub Actions task against an LM Studio OpenAI-compatible server."""

from __future__ import annotations

import argparse
import json
import sys
import urllib.error
import urllib.request


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Send a prompt to LM Studio's OpenAI-compatible chat API."


#### Analysis of `lm_studio_task.py`:
- **Objective**: Bridges GitHub Actions automated workflows with LM Studio local inference server using an OpenAI-compatible REST API (`/chat/completions`).
- **Key Features**: Built using pure Python standard libraries (`urllib.request`, `argparse`, `json`) without requiring external dependencies, making it lightweight for CI runners.

--- 
## 3. Hidden Files Discovery in NVIDIA CUTLASS Repository

Scanning the cloned `cutlass/` repository and workspace to locate all hidden files (files and directories starting with `.`).

In [12]:
import os

def scan_hidden_files(root_dir='.'):
    hidden_list = []
    root_path = Path(root_dir).resolve()
    
    for current_root, dirs, files in os.walk(root_path):
        rel_dir = Path(current_root).relative_to(root_path)
        # Exclude internal git blob object directory tree to keep output concise
        if '.git/objects' in str(rel_dir) or '.git/hooks' in str(rel_dir):
            continue
            
        for d in dirs:
            if d.startswith('.'):
                hidden_list.append({
                    'type': 'DIRECTORY',
                    'path': str(Path(current_root, d).relative_to(root_path))
                })
        for f in files:
            if f.startswith('.'):
                hidden_list.append({
                    'type': 'FILE',
                    'path': str(Path(current_root, f).relative_to(root_path))
                })
    return sorted(hidden_list, key=lambda x: x['path'])

cutlass_hidden = scan_hidden_files('.')
print(f"Total Hidden Items Found: {len(cutlass_hidden)}\n")
for item in cutlass_hidden:
    print(f"[{item['type']}] {item['path']}")

Total Hidden Items Found: 9

[DIRECTORY] .git
[DIRECTORY] .github
[FILE] .gitignore
[DIRECTORY] cutlass/.git
[DIRECTORY] cutlass/.github
[FILE] cutlass/.gitignore
[FILE] cutlass/.gitmodules
[FILE] cutlass/python/docs/.buildinfo
[FILE] cutlass/test/unit/nvrtc/thread/.gitignore
